# 미디어 콘텐츠 노출 지수 · 미디어 크로스오버 지수 — 최종 코퍼스 10,020건 검증 노트북

최종 데이터의 원본 지수 두 개를 코퍼스와 대조하고, v7 10·11라운드 병합 로그·K=9 검증·동결 LDA 미디어노출형 비중과 교차검증한다.

| 입력 | 내용 |
|---|---|
| `media_exposure_v7.json` | 4개 서브태그(예능/유튜브/영화/드라마) 키워드 매칭 — 1,025건(10.2%), 99개 팬덤 |
| `media_crossover_index_v7.json` | news_media 출처 도메인 기준 고유 매체 수·매체다양성비율 — 6,712건(67.0%), 고유 매체 1,298개 |
| `k9_validation_v7.json` | "독립적인 미디어 토픽/메타요인이 있는가" K=9 검증(보고서 10.8절) |
| `data/v7_rounds/merge_log_r10.json`, `merge_log_r11.json` | 미디어 크로스오버 리서치 라운드(131건 + 143건) |
| `fandom_scores_v6.json` | 동결 스냅샷(7,350건, K=10/M=5) — 보고서 본문 미디어노출형 비중 |

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

media = load_json(DATA_DIR / "media_exposure_v7.json")
xover = load_json(DATA_DIR / "media_crossover_index_v7.json")
k9 = load_json(DATA_DIR / "k9_validation_v7.json")
frozen_scores = load_json(DATA_DIR / "fandom_scores_v6.json")
fandoms = load_json(DATA_DIR / "fandoms_v3_100.json")
r10 = load_json(ROUNDS_DIR / "merge_log_r10.json")
r11 = load_json(ROUNDS_DIR / "merge_log_r11.json")
bullets_df = flatten_bullets(fandoms)
print("근거문장:", len(bullets_df))
print("media_exposure:", media["total_media_bullets"], f"({media['corpus_media_share']:.1%})", media["subtag_totals"], "| 1건 이상 팬덤", media["n_fandoms_with_any_media_bullet"])
print("media_crossover: news_media", xover["total_news_media_bullets"], f"({xover['corpus_news_media_share']:.1%})", "| 고유 매체", xover["n_distinct_outlets_corpuswide"])
print("r10:", r10["round"], r10["net_new_bullets"], "건 /", r10["n_touched_fandoms"], "팬덤 | r11:", r11["round"], r11["net_new_bullets"], "건 /", r11["n_touched_fandoms"], "팬덤")

근거문장: 10020
media_exposure: 1025 (10.2%) {'유튜브': 286, '예능': 309, '드라마': 313, '영화': 179} | 1건 이상 팬덤 99
media_crossover: news_media 6712 (67.0%) | 고유 매체 1298
r10: v7_r10_media_crossover_research 131 건 / 79 팬덤 | r11: v7_r11_media_crossover_reinforcement 143 건 / 68 팬덤


## 1. 데이터 무결성 검증 — 두 지수 모두

In [2]:
per_fandom_corpus = bullets_df.groupby("fandom").size().to_dict()
rows = []; n_mis = share_mis = 0
for rec in media["fandoms"]:
    if per_fandom_corpus.get(rec["fandom"]) != rec["n_total_bullets"]: n_mis += 1
    if abs(rec["n_media_bullets"] / rec["n_total_bullets"] - rec["media_share"]) > 0.0005: share_mis += 1
    row = {"팬덤": rec["fandom"], "구분": rec["category"], "근거문장수": rec["n_total_bullets"], "미디어문장수": rec["n_media_bullets"], "미디어비중": rec["media_share"]}
    row.update({s: rec["subtag_counts"].get(s, 0) for s in media["subtags"]})
    rows.append(row)
media_df = pd.DataFrame(rows)
sub_sum = {s: int(media_df[s].sum()) for s in media["subtags"]}
print(f"[노출] 근거문장 수 != 코퍼스: {n_mis} | media_share 불일치: {share_mis} | 합 {media_df['미디어문장수'].sum()} == {media['total_media_bullets']}: {media_df['미디어문장수'].sum() == media['total_media_bullets']} | 서브태그 합 == subtag_totals: {sub_sum == media['subtag_totals']}")

# 서브태그 키워드 재매칭(4개 서브태그명 그대로 부분 문자열) — 원본이 사용한 정확한 키워드 사전은 JSON에 없으므로 근사
for s in media["subtags"]:
    bullets_df[s] = bullets_df["text"].str.contains(s, regex=False)
bullets_df["media_hit"] = bullets_df[media["subtags"]].any(axis=1)
recomp = bullets_df.groupby("fandom")["media_hit"].sum().astype(int)
media_df["재매칭(서브태그명 단순 포함)"] = media_df["팬덤"].map(recomp)
diff = media_df["재매칭(서브태그명 단순 포함)"] - media_df["미디어문장수"]
print(f"[노출] 서브태그명 단순 포함 재매칭 합계 {int(recomp.sum())}건 vs 원본 {media['total_media_bullets']}건, 팬덤별 정확 일치 {(diff == 0).sum()}/100 (원본은 확장 키워드 사전 사용 — 근사 재현)")

rows = []; n_mis = ratio_mis = 0
for rec in xover["fandoms"]:
    if per_fandom_corpus.get(rec["fandom"]) != rec["n_total_bullets"]: n_mis += 1
    if rec["n_news_media_bullets"] and abs(rec["n_distinct_outlets"] / rec["n_news_media_bullets"] - rec["outlet_diversity_ratio"]) > 0.0005: ratio_mis += 1
    rows.append({"팬덤": rec["fandom"], "구분": rec["category"], "근거문장수": rec["n_total_bullets"], "뉴스매체문장수": rec["n_news_media_bullets"],
                 "뉴스매체비중": rec["news_media_share"], "고유매체수": rec["n_distinct_outlets"], "매체다양성비율": rec["outlet_diversity_ratio"],
                 "상위매체": ", ".join(o["domain"] for o in rec["top_outlets"][:3])})
xover_df = pd.DataFrame(rows)
print(f"[크로스오버] 근거문장 수 != 코퍼스: {n_mis} | outlet_diversity_ratio 불일치: {ratio_mis} | 뉴스매체문장 합 {xover_df['뉴스매체문장수'].sum()} == {xover['total_news_media_bullets']}: {xover_df['뉴스매체문장수'].sum() == xover['total_news_media_bullets']}")

[노출] 근거문장 수 != 코퍼스: 0 | media_share 불일치: 0 | 합 1025 == 1025: True | 서브태그 합 == subtag_totals: True
[노출] 서브태그명 단순 포함 재매칭 합계 1025건 vs 원본 1025건, 팬덤별 정확 일치 100/100 (원본은 확장 키워드 사전 사용 — 근사 재현)
[크로스오버] 근거문장 수 != 코퍼스: 0 | outlet_diversity_ratio 불일치: 0 | 뉴스매체문장 합 6712 == 6712: True


## 2. 팬덤별 미디어 노출 — 상위 15 (미디어문장수 순) / 매체 확산 폭 상위 10

In [3]:
media_df = media_df.sort_values(["미디어문장수", "미디어비중"], ascending=[False, False]).reset_index(drop=True); media_df.index += 1
for name in ["BTS", "임영웅", "리센느(RESCENE)"]:
    r = media_df[media_df["팬덤"] == name]; x = xover_df[xover_df["팬덤"] == name]
    print(f"{name}: 미디어 {int(r['미디어문장수'].iloc[0])}건 ({r['미디어비중'].iloc[0]:.1%}, {r.index[0]}위) | 고유매체 {int(x['고유매체수'].iloc[0])}개, 매체다양성비율 {x['매체다양성비율'].iloc[0]:.3f}")
display(media_df.head(15))
xover_df.sort_values("고유매체수", ascending=False).head(10).reset_index(drop=True)

BTS: 미디어 13건 (5.7%, 28위) | 고유매체 96개, 매체다양성비율 0.542
임영웅: 미디어 27건 (20.6%, 1위) | 고유매체 60개, 매체다양성비율 0.496
리센느(RESCENE): 미디어 10건 (11.8%, 44위) | 고유매체 36개, 매체다양성비율 0.529


,팬덤,구분,근거문장수,미디어문장수,미디어비중,예능,유튜브,영화,드라마,재매칭(서브태그명 단순 포함)
1,임영웅,트로트,131,27,0.2061,4,11,8,5,27
2,카더가든,솔로,74,26,0.3514,14,10,3,2,26
3,오마이걸,K-pop 걸그룹,98,21,0.2143,9,3,3,9,21
4,SEVENTEEN,K-pop 보이그룹,157,20,0.1274,7,4,4,7,20
5,EXO,K-pop 보이그룹,121,19,0.1570,6,0,8,10,19
6,엄정화,솔로,71,17,0.2394,2,2,12,2,17
7,성시경,발라드,80,17,0.2125,5,9,0,3,17
8,다비치,발라드,80,17,0.2125,0,1,0,16,17
9,츄,솔로,93,17,0.1828,4,7,0,7,17
10,이찬원,트로트,106,17,0.1604,6,9,0,2,17


,팬덤,구분,근거문장수,뉴스매체문장수,뉴스매체비중,고유매체수,매체다양성비율,상위매체
0,BTS,K-pop 보이그룹,227,177,0.7797,96,0.5424,"starnewskorea.com, v.daum.net, forbes.com"
1,TWICE,K-pop 걸그룹,191,134,0.7016,90,0.6716,"infobae.com, allkpop.com, natalie.mu"
2,BLACKPINK,K-pop 걸그룹,195,139,0.7128,83,0.5971,"thairath.co.th, infobae.com, finance.sina.com.cn"
3,지드래곤 (G-Dragon),힙합,133,128,0.9624,80,0.6250,"v.daum.net, newsis.com, sports.khan.co.kr"
4,싸이,솔로,122,97,0.7951,76,0.7835,"v.daum.net, cbc.ca, imaeil.com"
5,(여자)아이들,K-pop 걸그룹,118,84,0.7119,75,0.8929,"star.ettoday.net, nownews.com, m.joseilbo.com"
6,BABYMONSTER,K-pop 걸그룹,132,93,0.7045,72,0.7742,"barks.jp, oricon.co.jp, thairath.co.th"
7,LE SSERAFIM,K-pop 걸그룹,143,108,0.7552,71,0.6574,"oricon.co.jp, infobae.com, idntimes.com"
8,aespa,K-pop 걸그룹,158,94,0.5949,69,0.7340,"infobae.com, marieclairekorea.com, smentertainment.com"
9,NewJeans,K-pop 걸그룹,154,114,0.7403,67,0.5877,"harpersbazaar.co.kr, allkpop.com, news.qq.com"


## 3. 코퍼스 증거와의 교차검증 — v7 10·11라운드 미디어 크로스오버 리서치 신규 근거 수와의 관계

In [4]:
combined_new = {}
for log in (r10, r11):
    for fandom, n in log["per_group_added"].items():
        combined_new[fandom] = combined_new.get(fandom, 0) + n
media_df["r10+r11 신규 근거"] = media_df["팬덤"].map(combined_new).fillna(0).astype(int)
tot = sum(combined_new.values())
print(f"r10+r11 신규 근거 합 {tot}건 == {r10['net_new_bullets']}+{r11['net_new_bullets']}={r10['net_new_bullets'] + r11['net_new_bullets']} :", tot == r10["net_new_bullets"] + r11["net_new_bullets"])
r_cnt = np.corrcoef(media_df["r10+r11 신규 근거"], media_df["미디어문장수"])[0, 1]
r_shr = np.corrcoef(media_df["r10+r11 신규 근거"], media_df["미디어비중"])[0, 1]
print(f"Pearson r(신규 근거 수, 미디어문장수) = {r_cnt:.4f} | r(신규 근거 수, 미디어비중) = {r_shr:.4f}")
print("-> 키워드 지수는 그 라운드에서 들어온 문장을 직접 세므로 신규 건수와 양의 상관이 나오는 것이 자연스럽다(비중은 팬덤 규모로 나눠 상관이 약해진다).")
top_new = set(media_df.sort_values("r10+r11 신규 근거", ascending=False).head(10)["팬덤"])
top_cnt = set(media_df.head(10)["팬덤"])
print(f"신규 근거 상위 10 ∩ 미디어문장수 상위 10 = {len(top_new & top_cnt)}개: {sorted(top_new & top_cnt)}")

r10+r11 신규 근거 합 274건 == 131+143=274 : True
Pearson r(신규 근거 수, 미디어문장수) = 0.4259 | r(신규 근거 수, 미디어비중) = 0.2358
-> 키워드 지수는 그 라운드에서 들어온 문장을 직접 세므로 신규 건수와 양의 상관이 나오는 것이 자연스럽다(비중은 팬덤 규모로 나눠 상관이 약해진다).
신규 근거 상위 10 ∩ 미디어문장수 상위 10 = 2개: ['이찬원', '임영웅']


## 4. LDA 미디어노출형 비중(동결 스냅샷, 보고서 본문)과 키워드 지수의 대조 + K=9 검증 요약

In [5]:
def check_factor_share(scores, label):
    sum_mismatch, dominant_mismatch = [], []
    for d in scores:
        shares = d["factor_share"]
        if abs(sum(shares.values()) - 1.0) > 0.001:
            sum_mismatch.append((d["fandom"], round(sum(shares.values()), 4)))
        if max(shares, key=shares.get) != d["dominant_factor"]:
            dominant_mismatch.append((d["fandom"], max(shares, key=shares.get), d["dominant_factor"]))
    print(f"[{label}] factor_share 합 != 1.0 인 팬덤 수: {len(sum_mismatch)} / {len(scores)}")
    print(f"[{label}] dominant_factor 재계산 불일치 팬덤 수: {len(dominant_mismatch)} / {len(scores)}")
    return sum_mismatch, dominant_mismatch

check_factor_share(frozen_scores, "동결 7,350건 K=10/M=5")
FACTOR = "미디어노출형(방송·조회수)"
media_df["LDA 미디어노출형(동결)"] = media_df["팬덤"].map({d["fandom"]: d["factor_share"].get(FACTOR, 0.0) for d in frozen_scores})
sub = media_df.dropna(subset=["LDA 미디어노출형(동결)"])
print(f"n={len(sub)}, Pearson r(미디어비중, LDA 미디어노출형)={np.corrcoef(sub['미디어비중'], sub['LDA 미디어노출형(동결)'])[0, 1]:.4f}, "
      f"Spearman rho={sub[['미디어비중', 'LDA 미디어노출형(동결)']].corr(method='spearman').iloc[0, 1]:.4f}")
print("동결에서 dominant_factor=미디어노출형:", [d["fandom"] for d in frozen_scores if d["dominant_factor"] == FACTOR])
print()
print("K=9 검증(k9_validation_v7.json):", k9["methodology"][:90], "...")
print("  corpus_docs:", k9["corpus_docs"], "| K-grid 승자:", k9["k_grid_winner"], "| 기존 그리드에 K=9 없었음:", k9["existing_k_grid_never_tested_k9"])
print("  K=9 phi의 M-grid에서 media 토픽이 단독 메타요인으로 분리되는 M:", [m["m"] for m in k9["m_grid_on_k9_phi"] if m["media_topic_isolated"]],
      "| 각 M 실루엣:", {m["m"]: m["silhouette"] for m in k9["m_grid_on_k9_phi"]})
print("  서브태그 문서빈도(%):", {k: v["pct_of_docs"] for k, v in k9["subtag_doc_freq"].items()})
print("  -> 미디어 토픽이 분리되는 M=6~8에서 실루엣이 0.08 이하로 떨어져 통계적으로 독립된 미디어 메타요인은 채택되지 않았고, 키워드 지수(media_exposure_v7.json)로 대체됐다.")
pd.DataFrame(k9["k_grid"])

[동결 7,350건 K=10/M=5] factor_share 합 != 1.0 인 팬덤 수: 0 / 100
[동결 7,350건 K=10/M=5] dominant_factor 재계산 불일치 팬덤 수: 0 / 100
n=97, Pearson r(미디어비중, LDA 미디어노출형)=0.5196, Spearman rho=0.4688
동결에서 dominant_factor=미디어노출형: ['박정현', '브라운아이즈']

K=9 검증(k9_validation_v7.json): run_lda_v6.py와 동일 코퍼스/토크나이저/벡터라이저(min_df=2,max_df=0.6)로 재현. K-grid에 K=9를 추가해 재학습(perplexit ...
  corpus_docs: 9614 | K-grid 승자: {'k': 8, 'perplexity': 3413.1, 'coherence': -2.385, 'diversity': 0.825, 'stability': 0.508, 'composite_rank_sum': 9} | 기존 그리드에 K=9 없었음: True
  K=9 phi의 M-grid에서 media 토픽이 단독 메타요인으로 분리되는 M: [6, 7, 8] | 각 M 실루엣: {4: 0.064, 5: 0.099, 6: 0.081, 7: 0.042, 8: 0.023}
  서브태그 문서빈도(%): {'예능': 2.92, '유튜브': 2.83, '영화': 1.52, '드라마': 2.89}
  -> 미디어 토픽이 분리되는 M=6~8에서 실루엣이 0.08 이하로 떨어져 통계적으로 독립된 미디어 메타요인은 채택되지 않았고, 키워드 지수(media_exposure_v7.json)로 대체됐다.


,k,perplexity,coherence,diversity,stability,composite_rank_sum
0,8,3413.1,-2.385,0.825,0.508,9
1,9,3454.5,-2.430,0.844,0.429,13
2,10,3458.0,-2.448,0.850,0.412,15
3,12,3349.5,-2.293,0.825,0.383,10
4,15,3413.3,-2.427,0.793,0.366,17
5,20,3568.7,-2.498,0.765,0.356,26
6,25,3666.5,-2.374,0.776,0.333,22
7,30,3775.4,-2.609,0.747,0.321,32


## 5. 한계

1. 두 지수 모두 LDA와 무관한 문자열/도메인 매칭 보조지표다. 크로스오버 지수는 "팬 유입 경로"를 측정하지 않으며 매체 확산 폭의 대리치일 뿐이다(`scope_and_limits`).
2. 노출 지수의 확장 키워드 사전은 JSON에 없어 1절 재매칭은 서브태그명 단순 포함으로만 근사했다.
3. `domain_of()`/`source_type_of()`는 최종 토크나이저 소스(`run_lda_v6_live_reference_v7.py`, 저장소에 없음)의 정의를 쓴 것이므로 고유 매체 수는 원본 값을 그대로 검증 대상으로 삼았다.